Import the relevant python modules required for the example: xarray and matplotlib

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from SubRoutines.plotting import LccsManager

Specify the file to read in. In this case we are reading LST derived from MODIS AQUA, the data has been process to L3 and therefore has been reprojected to a regular grid (0.25 degrees in this example): 

In [ ]:
esacci_lst_025 = "Data/ESACCI-LST-L3C-LST-MODISA-0.25deg_1MONTHLY_DAY-20060701000000-fv3.00.nc"

Open file using xarray (xr), the file will be read into a dataset (ds) structure:

In [ ]:
ds = xr.load_dataset(esacci_lst_025)

We can display the data within the dataset easily:

In [ ]:
ds

We can also plot the lst data directly (size and aspect are added here to make the image larger, but are not required):

In [ ]:
ds.lst.plot(aspect=2,size=8)

We can also look at the global distribution of LST:

In [ ]:
ds.lst.plot.hist(size=8,bins=50, alpha=0.5, color='steelblue',edgecolor='grey')
;

And we can look at specifc regions, first lets set our region of interest: 

In [ ]:
lower_lon = 105.0
upper_lon = 145.0
lower_lat = 25.0
upper_lat = 65.0

xarray can select data by coordinate value, so there is no need to work out array indexes ourselves. Using `.sel` with a `slice` of lat and lon values cuts out the region of interest (the bounds are inclusive; this assumes the coordinates are in ascending order, which they are for these files):

In [ ]:
region = ds.sel(lat=slice(lower_lat, upper_lat), lon=slice(lower_lon, upper_lon))

Ready to go. The plotting code is now simply run on the selected region:

In [ ]:
region.lst.plot(aspect=2,size=8)

We can also run for transects. We specify a latitude and use `method="nearest"` to select the closest grid row, then plot all longitudes at that latitude:

In [ ]:
lat_transect = 50.0
ds.lst.sel(lat=lat_transect, method="nearest").plot(aspect=2,size=8)

We can also explore some of the other paramters provided with the LST CCI data by looking at another product. Here we will use some 0.01 degree MODIS AQUA data that has been regridded to a subset (lat: 40-->45, lon: 117-->122):

In [ ]:
esacci_lst_001 = "Data/ESACCI-LST-L3C-LST-MODISA-0.01deg_1MONTHLY_DAY-20220901103756-fv3.00.nc"
regridded_ds = xr.load_dataset(esacci_lst_001)

In [ ]:

regridded_ds.lst.plot(aspect=2,size=8)
plt.title(f"time = {np.datetime_as_string(regridded_ds.time.values[0],unit='M')}")
;

The data contains several variables that were not in the 0.25 degree data such as land cover classification (lcc):

In [ ]:
lccs_data = LccsManager(regridded_ds)
lccs_data.analyse_region()
lccs_data.plot_map()

We can inspect the corresponding values of these classes in the attributes:

In [ ]:
regridded_ds.lcc.attrs

We can also inspect the distribution of these lccs:

Note the plot currently displays as a percentage cover of the region interest. If you would prefer the raw number of pixels in each class change "percentage=True" to "precentage=False".

In [ ]:
lccs_data.plot_distribution(percentage=True)

This can be correlated against LST to see the lcc distribution of temperature.

In [ ]:
lccs_data.bin_lst_by_lcc()
lccs_data.box_plot()

xarray can also calculate grouped statistics directly, without any manual binning. Grouping the LST by land cover class gives the mean, standard deviation and number of pixels for each class:

In [ ]:
by_lcc = regridded_ds.lst.groupby(regridded_ds.lcc)
summary = xr.Dataset({"mean": by_lcc.mean(), "std": by_lcc.std(), "count": by_lcc.count()})

labels = lccs_data.legend.set_index("NB_LAB").LCCOwnLabel
summary.to_dataframe().rename(index=labels).round(2)